<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/miso_direc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
import time
import copy

# ── HELPERS classification (repris du programme binarisé) ────────────────────
def cls_acc(f, y):
    sg = np.sign(f)
    return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
def cls_auc(f, y):
    try: return float(roc_auc_score((y > 0).astype(int), f))
    except Exception: return float('nan')
def cls_str(f, y):
    return f"acc={cls_acc(f,y):.4f} AUC={cls_auc(f,y):.4f}"

params = {
    "n_ambiant": 4, "deg_P": 3, "n_terms_poly": 20,
    "seeds": {1: 3709453, 2: 2504, 3: 54205,
              42: 59412, 43: 6826, 44: 94403, 11: 6244, 12: 241},
    "weights": {0: 0, 1: 1, 2: 0.1, 3: 0.01},

    "n_train": 50, "n_test": 5000,

    # ── cascade gloutonne "pure" (un sigma GLOBAL par niveau, décroissant) ──
    "levels": 15, "sigma0": 1.5, "sigma_decay": 0.9,
    "n_dict_candidates": 2000, "n_centers_per_level": 800,
    "n_stop": 1e3, "plateau_window": 14, "plateau_tol": 1e-6,
    "lambda_reg": 1e-2,
    "n_G": 2000,          # points cloud pour le Gram Sobolev de chaque niveau

    # ── directionnel ──
    "n_dirs": 50,          # directions aléatoires indépendantes par point du nuage
    "batch_dirs": 10,      # taille des blocs de directions (mémoire de crête ~ n*m*batch_dirs)
}
params["n_unlabeled"] = np.maximum(4000 - params["n_train"], 500)
params["train_center_ratio"] = 0.5 + params["n_train"] / (2 * 4000)

# ══════════════════════════════════════════════════════════════════════════════
# POLYNÔMES ET CLOUD SUR {Q=0}  (repris tel quel, code déjà validé)
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree+1), repeat=d) if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: term *= x[:, j]**e
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices) if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_c = np.max(np.abs(coeffs))
    if max_c > 0:
        nc = coeffs / max_c
        def Pn(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(nc, indices):
                term = np.ones(x.shape[0])
                for j, e in enumerate(alpha):
                    if e > 0: term *= x[:, j]**e
                y += c * term
            return y
        return Pn
    return P

P1, i1, c1 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][1])
P2, i2, c2 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][2])
P3, i3, c3 = random_sparse_polynomial(params["n_ambiant"], params["deg_P"], params["n_terms_poly"], params["seeds"][3])
P1 = normalize_polynomial(P1, i1, c1); P2 = normalize_polynomial(P2, i2, c2); P3 = normalize_polynomial(P3, i3, c3)
def Q(x): return P1(x)*P2(x)*P3(x)

def grad_Q_analytical(X):
    eps = 1e-5; d = X.shape[1]; p1 = P1(X); p2 = P2(X); p3 = P3(X)
    grad = np.zeros_like(X)
    for k in range(d):
        Xp = X.copy(); Xp[:, k] += eps; Xm = X.copy(); Xm[:, k] -= eps
        dp1 = (P1(Xp)-P1(Xm))/(2*eps); dp2 = (P2(Xp)-P2(Xm))/(2*eps); dp3 = (P3(Xp)-P3(Xm))/(2*eps)
        grad[:, k] = dp1*p2*p3 + p1*dp2*p3 + p1*p2*dp3
    return grad

def project_to_Q_zero(X_init, n_steps=60, tol=1e-4, damp=1.0):
    X = X_init.copy()
    for _ in range(n_steps):
        q = Q(X); gq = grad_Q_analytical(X)
        g2 = np.sum(gq*gq, axis=1, keepdims=True) + 1e-12
        X = X - damp*(q[:, None]*gq)/g2
        X = np.clip(X, 0., 1.)
        if np.abs(Q(X)).max() < tol*0.1:
            break
    return X, np.abs(Q(X)) < tol

def sample_on_Q_zero(n_target, d, seed=None, max_batches=200):
    rng = np.random.default_rng(seed)
    collected = []; n_col = 0; n_seen = 0; n_ok = 0
    for _ in range(max_batches):
        if n_col >= n_target:
            break
        n_batch = min(max((n_target - n_col)*4, 200), 20000)
        Xi = rng.uniform(0, 1, (int(n_batch), d))
        Xp, conv = project_to_Q_zero(Xi)
        good = Xp[conv]
        good = good[np.all(np.isfinite(good), axis=1)]
        n_seen += len(Xi); n_ok += int(conv.sum())
        if len(good) > 0:
            collected.append(good); n_col += len(good)
        print(f"  collectés:{n_col}/{n_target} (cumulé {n_ok/max(n_seen,1):.1%})", end='\r')
    print()
    if n_col < n_target:
        raise RuntimeError(f"sample_on_Q_zero: seulement {n_col}/{n_target} points.")
    return np.vstack(collected)[:n_target]

print("Génération du cloud sur {Q=0}...")
d = params["n_ambiant"]
X_train = sample_on_Q_zero(params["n_train"], d, seed=params["seeds"][42])
X_test = sample_on_Q_zero(params["n_test"], d, seed=params["seeds"][43])
X_unlabeled = sample_on_Q_zero(params["n_unlabeled"], d, seed=params["seeds"][44])
print(f"  X_train:{X_train.shape}  X_unlabeled:{X_unlabeled.shape}")

P_target1, indicest1, coeffst1 = random_sparse_polynomial(params["n_ambiant"], 4, params["n_terms_poly"], seed=params["seeds"][11])
P_target2, _, _ = random_sparse_polynomial(params["n_ambiant"], 4, params["n_terms_poly"], seed=params["seeds"][12])
Ptarget1 = normalize_polynomial(P_target1, indicest1, coeffst1)
Ptarget2 = normalize_polynomial(P_target2, indicest1, coeffst1)

def target_function(X):
    return np.minimum(np.abs(P_target1(X)), 1) + np.minimum(np.abs(P_target2(X)), 1)

y_cont_train = target_function(X_train); y_cont_test = target_function(X_test)
X_all = np.vstack([X_train, X_unlabeled]); N_all = len(X_all)

_med = float(np.median(target_function(X_unlabeled)))
y_train = np.where(y_cont_train >= _med, 1.0, -1.0)
y_test = np.where(y_cont_test >= _med, 1.0, -1.0)
print(f"  cible binarisée au seuil médian={_med:.4f} : "
      f"train (+1)={int((y_train>0).sum())}/{len(y_train)}  "
      f"test (+1)={int((y_test>0).sum())}/{len(y_test)}")
_bal_te = (y_test > 0).mean()
if _bal_te < 0.3 or _bal_te > 0.7:
    print(f"  [!] classes déséquilibrées ({_bal_te:.1%} de +1) — préférer l'AUC.")


# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES ISOTROPES DIRECTIONNELLES — sigma SCALAIRE unique par niveau
# (cascade gloutonne "pure" : un seul sigma partagé par tous les centres de ce
# niveau, décroissant géométriquement — PAS le style miso à sigma aléatoire
# par centre). Dérivées directionnelles le long de n_dirs directions ALÉATOIRES
# INDÉPENDANTES PAR POINT du nuage. Formules exactes (calibration + correction
# par trace/grad-Laplacien) — validées contre le calcul dense à ~2-3% près
# (bruit MC pur, décroît avec n·n_dirs).
# ══════════════════════════════════════════════════════════════════════════════
def gaussian_features(X, centers, sigma):
    diff = X[:, None, :] - centers[None, :, :]
    sq = np.sum(diff**2, axis=2)
    return np.exp(-sq/(2*sigma**2))

def build_G_directional_streaming(X_cloud, centers, sigma, weights, n_dirs,
                                  batch_dirs=10, rng=None):
    """Construit G directement, PAR BLOCS de `batch_dirs` directions — jamais
    les n_dirs directions en mémoire simultanément (c'est ça qui faisait
    exploser la RAM). TR et V (Laplacien, grad-Laplacien) ne dépendent PAS des
    directions : calculés UNE SEULE FOIS, avant la boucle sur les blocs.
    Résultat identique (à bruit MC près, ~2-3% pour n_dirs modeste) à la
    version qui construisait D1/D2/D3 (n,m,n_dirs) d'un coup."""
    if rng is None:
        rng = np.random.default_rng(0)
    n, d_ = X_cloud.shape
    m = centers.shape[0]
    diff = X_cloud[:, None, :] - centers[None, :, :]     # (n,m,d) — calculé une fois
    sq = np.sum(diff**2, axis=2)
    phi = np.exp(-sq/(2*sigma**2))                       # (n,m)

    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    w2 = weights.get(2, 0.); w3 = weights.get(3, 0.)
    need2 = w2 != 0; need3 = w3 != 0

    G = np.zeros((m, m))
    if w0:
        G += w0*(phi.T@phi)/n

    if need2:
        # Laplacien exact — indépendant des directions
        TR = (-d_/sigma**2 + sq/sigma**4)*phi
        TRG = (TR.T@TR)/n
    if need3:
        # grad(Laplacien) exact — indépendant des directions, O(n·m·d)
        A = -d_/sigma**2; B = 1/sigma**4
        coefV = 2*B - (A/sigma**2) - (B/sigma**2)*sq
        V = phi[:, :, None]*diff*coefV[:, :, None]
        VVG = np.einsum('nid,njd->ij', V, V)/n

    MC1_sum = np.zeros((m, m)); MC2_sum = np.zeros((m, m)); MC3_sum = np.zeros((m, m))
    done = 0
    while done < n_dirs:
        K = min(batch_dirs, n_dirs - done)
        U = rng.normal(size=(n, K, d_))
        U /= np.linalg.norm(U, axis=2, keepdims=True)
        a = np.einsum('nmd,nkd->nmk', diff, U)            # (n,m,K) — SEUL ce bloc en mémoire
        gp = -a/sigma**2
        if w1:
            D1 = gp*phi[:, :, None]
            D1f = D1.transpose(0, 2, 1).reshape(-1, m)
            MC1_sum += D1f.T@D1f
            del D1, D1f
        if need2 or need3:
            gpp = -1/sigma**2
            D2 = (gpp + gp**2)*phi[:, :, None]
            if need2:
                D2f = D2.transpose(0, 2, 1).reshape(-1, m)
                MC2_sum += D2f.T@D2f
                del D2f
            if need3:
                D3 = (3*gp*gpp + gp**3)*phi[:, :, None]
                D3f = D3.transpose(0, 2, 1).reshape(-1, m)
                MC3_sum += D3f.T@D3f
                del D3, D3f
            del D2
        del U, a, gp                                       # jamais accumulé — bloc suivant repart de zéro
        done += K

    if w1:
        G += w1*d_*MC1_sum/(n*n_dirs)
    if need2:
        MC2 = MC2_sum/(n*n_dirs)
        G += w2*(d_*(d_+2)*MC2 - TRG)/2
    if need3:
        MC3 = MC3_sum/(n*n_dirs)
        G += w3*(d_*(d_+2)*(d_+4)*MC3 - 9*VVG)/6
    return G
    return G

def sample_candidates(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    n_tr = min(int(round(n_candidates*train_ratio)), X_train.shape[0])
    n_ul = min(n_candidates-n_tr, X_unlabeled.shape[0]); parts = []
    if n_tr > 0: parts.append(X_train[rng.choice(X_train.shape[0], n_tr, replace=False)])
    if n_ul > 0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0], n_ul, replace=False)])
    return np.vstack(parts)


# ══════════════════════════════════════════════════════════════════════════════
# CASCADE GLOUTONNE PURE (sigma décroissant, directionnel, binarisé)
# ══════════════════════════════════════════════════════════════════════════════
losses = []; times = []
pred_train_accum = np.zeros(len(X_train)); pred_test_accum = np.zeros(len(X_test))
history = []; stop_reason = None; final_level = None

for level in range(params["levels"]):
    t0 = time.time()
    sigma = params["sigma0"]*params["sigma_decay"]**level
    lambda_reg = max(params["lambda_reg"], 0)
    rng = np.random.default_rng(seed=level)

    candidates = sample_candidates(X_train, X_unlabeled, params["n_dict_candidates"],
                                   params["train_center_ratio"], rng)
    A_cand = gaussian_features(X_train, candidates, sigma)
    residual = y_train - pred_train_accum
    corr = (A_cand.T@residual)/len(X_train)
    scores = corr**2

    k = min(params["n_centers_per_level"], len(candidates))
    top_k = np.argsort(scores)[-k:]
    centers = candidates[top_k]; A = A_cand[:, top_k]
    Atest = gaussian_features(X_test, centers, sigma)

    n_G = min(N_all, params["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G_directional_streaming(X_all[idx_G], centers, sigma, params["weights"],
                                      n_dirs=params["n_dirs"], batch_dirs=params["batch_dirs"],
                                      rng=np.random.default_rng(9000+level))

    n = A.shape[0]
    M = (A.T@A)/n + lambda_reg*G
    rhs = (A.T@residual)/n
    eigvals, eigvecs = np.linalg.eigh(M)
    thresh = max(lambda_reg, 1e-7); mask = eigvals > thresh
    V = eigvecs[:, mask]; S = eigvals[mask]; coeffs = V@((V.T@rhs)/S)

    loss_data = np.mean((residual - A@coeffs)**2)
    loss_reg = lambda_reg*coeffs@G@coeffs
    loss_total = loss_data + loss_reg
    t1 = time.time(); times.append(t1-t0)

    print(f"\n=== LEVEL {level} | σ={sigma:.4f} | λ={lambda_reg:.2e} | "
          f"dict={len(candidates)} → k={k} | n_G={n_G} ===")
    print(f"  rang={mask.sum()}/{k}  temps={times[-1]:.1f}s")
    print(f"  loss={loss_total:.6f}  (data={loss_data:.6f}  reg={loss_reg:.6f})")

    if len(history) > 0:
        prev_loss = history[-1]['loss_total']
        if loss_total > prev_loss + params["n_stop"]*lambda_reg:
            best_idx = len(history)-1
            for i in range(len(history)-2, -1, -1):
                if history[i]['loss_total'] <= history[i+1]['loss_total']: best_idx = i
                else: break
            print(f"  ⚠ ARRÊT : explosion détectée -> on garde le niveau {history[best_idx]['level']+1}")
            stop_reason = "explosion"; final_level = best_idx; break

    pred_train_accum_new = pred_train_accum + A@coeffs
    pred_test_accum_new = pred_test_accum + Atest@coeffs

    print(f"  {cls_str(pred_test_accum_new, y_test)}")
    losses.append(loss_total)
    pred_train_accum = pred_train_accum_new; pred_test_accum = pred_test_accum_new
    history.append({'level': level, 'pred_train': pred_train_accum.copy(),
                    'pred_test': pred_test_accum.copy(), 'loss_total': loss_total})

    w = params["plateau_window"]
    if len(history) >= w:
        recent = [h['loss_total'] for h in history[-w:]]
        improvement = recent[0]-recent[-1]
        if improvement < params["plateau_tol"]:
            print(f"  ⚠ ARRÊT : plateau détecté (baisse sur {w} niveaux = {improvement:.6f})")
            stop_reason = "plateau"; final_level = len(history)-w
            break

if stop_reason is None:
    final_level = len(history)-1; stop_reason = "max_levels"
final_state = history[final_level]
pred_test_final = final_state['pred_test']

print("\n" + "="*80)
print(f"ARRÊT : raison = {stop_reason}")
print(f"Niveau final retenu : {final_state['level']+1} / {params['levels']} effectués")
print(f"{cls_str(pred_test_final, y_test)}")
print("="*80)


# ══════════════════════════════════════════════════════════════════════════════
# COMPARAISONS : Ridge polynomial, SVM RBF, Ridge RBF  (pas de Phase 2)
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.svm import SVC
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

print("\nCalibration Ridge polynomial...")
poly = PolynomialFeatures(degree=min(params["deg_P"], 8), include_bias=False)
ridge_poly = Ridge(alpha=1e-8)
ridge_poly.fit(poly.fit_transform(X_train), y_train)
pred_ridgepoly_te = ridge_poly.predict(poly.transform(X_test))

print("Calibration SVM RBF (GridSearchCV)...")
param_grid_svm = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1, 1]}
svm = GridSearchCV(SVC(kernel="rbf"), param_grid_svm, cv=5, n_jobs=-1)
svm.fit(X_train, y_train)
pred_svm_te = svm.decision_function(X_test)

print("Calibration Ridge RBF (GridSearchCV)...")
param_grid_kr = {"alpha": [1e-3, 1e-2, 1e-1, 1.0], "gamma": [0.001, 0.01, 0.1, 1]}
kr = GridSearchCV(KernelRidge(kernel="rbf"), param_grid_kr, cv=5, n_jobs=-1)
kr.fit(X_train, y_train)
pred_kr_te = kr.predict(X_test)

print("\n" + "="*80)
print("RÉSUMÉ")
print("="*80)
print(f"Sobolev directionnel (glouton) : {cls_str(pred_test_final, y_test)}  "
      f"(niveau {final_state['level']+1}, arrêt={stop_reason})")
print(f"Ridge polynomial               : {cls_str(pred_ridgepoly_te, y_test)}")
print(f"SVM RBF     (best={svm.best_params_})  : {cls_str(pred_svm_te, y_test)}")
print(f"Ridge RBF   (best={kr.best_params_})  : {cls_str(pred_kr_te, y_test)}")
print(f"\nweights={params['weights']}  λ={params['lambda_reg']}  n_dirs={params['n_dirs']}")
print(f"n_train={params['n_train']}  n_unlabeled={params['n_unlabeled']}  n_test={params['n_test']}")
print(f"temps cascade : {sum(times):.1f}s total ({np.mean(times):.1f}s/niveau, {len(times)} niveaux)")
print("\nDécroissance de la loss par niveau :")
for i, l in enumerate(losses):
    marker = "  ← retenu" if i == final_level else ""
    print(f"  Niveau {i+1:2d} : {l:.6f}  ({times[i]:.1f}s){marker}")

Génération du cloud sur {Q=0}...
  collectés:200/50 (cumulé 100.0%)
  collectés:20000/5000 (cumulé 100.0%)
  collectés:15800/3950 (cumulé 100.0%)
  X_train:(50, 4)  X_unlabeled:(3950, 4)
  cible binarisée au seuil médian=1.0768 : train (+1)=23/50  test (+1)=2545/5000

=== LEVEL 0 | σ=1.5000 | λ=1.00e-03 | dict=2000 → k=800 | n_G=2000 ===
  rang=5/800  temps=39.4s
  loss=0.665334  (data=0.660185  reg=0.005148)
  acc=0.7044 AUC=0.7949

=== LEVEL 1 | σ=1.3500 | λ=1.00e-03 | dict=2000 → k=800 | n_G=2000 ===
  rang=6/800  temps=38.5s
  loss=0.599768  (data=0.596666  reg=0.003102)
  acc=0.7204 AUC=0.7992

=== LEVEL 2 | σ=1.2150 | λ=1.00e-03 | dict=2000 → k=800 | n_G=2000 ===
  rang=7/800  temps=37.3s
  loss=0.523443  (data=0.519054  reg=0.004389)
  acc=0.7392 AUC=0.7836

=== LEVEL 3 | σ=1.0935 | λ=1.00e-03 | dict=2000 → k=800 | n_G=2000 ===
  rang=10/800  temps=46.7s
  loss=0.447936  (data=0.441771  reg=0.006166)
  acc=0.7256 AUC=0.7740

=== LEVEL 4 | σ=0.9842 | λ=1.00e-03 | dict=2000 → k=80